# Validation

How do we know the estimator is correct and not 'a big pile of code that
returns nonsense'? **Three** independent checks, all run live in this
notebook.

1. **No-censoring limit:** with thresholds pushed beyond the data range,
   no observation is censored, so the censored/truncated MLE *must*
   reduce to OLS. Verified against `statsmodels.OLS`.
2. **Reference packages:** the test suite also compares against R's
   `survival::survreg` (the engine behind `AER::tobit`) and `truncreg`
   when R is available; see `tests/test_r_reference.py`. We reproduce
   that R comparison live below.
3. **Probit limit:** with $L = R = c$, the censored log-likelihood is
   *exactly* the probit log-likelihood on $\mathbf 1\{Y^* > c\}$, and
   the relation $\beta_{\text{tobit}}/\sigma_{\text{tobit}} =
   \beta_{\text{probit}}$ holds (Hansen 2022, §27.4). A narrow band
   $L = c - \varepsilon, R = c + \varepsilon$ is enough to verify it
   numerically without making $\sigma$ unidentified.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import CensoredRegression, TruncatedRegression

rng = np.random.default_rng(0)
n = 1000
X = rng.normal(size=(n, 3))
y = 2.0 + 1.5*X[:,0] - 0.7*X[:,1] + 0.3*X[:,2] + rng.normal(scale=1.3, size=n)

## No censoring $\Rightarrow$ OLS

We set `left=-1e6`, `right=1e6` so that no observation is censored. The censored log-likelihood then reduces to the Gaussian log-likelihood, whose maximiser is exactly the OLS coefficient vector.

In [2]:
ols = sm.OLS(y, sm.add_constant(X)).fit()
cens = CensoredRegression(left=-1e6, right=1e6).fit(X, y)
trunc = TruncatedRegression(left=-1e6, right=1e6).fit(X, y)

pd.DataFrame({
    'OLS':            np.asarray(ols.params),
    'Censored MLE':   cens.coef_,
    'Truncated MLE':  trunc.coef_,
}, index=['const', 'x1', 'x2', 'x3']).round(6)

,OLS,Censored MLE,Truncated MLE
const,2.046512,2.046482,2.046512
x1,1.434191,1.434221,1.434191
x2,-0.626598,-0.626603,-0.626598
x3,0.345980,0.345982,0.345980


The columns agree to several decimals. We can quantify the maximum discrepancy and confirm the scale parameter and log-likelihood match too (using the MLE scale $\hat\sigma = \sqrt{\mathrm{SSR}/n}$).

In [3]:
ols_beta = np.asarray(ols.params)
print(f'max |beta_censored - beta_OLS|  = {np.max(np.abs(cens.coef_ - ols_beta)):.2e}')
print(f'max |beta_truncated - beta_OLS| = {np.max(np.abs(trunc.coef_ - ols_beta)):.2e}')
print(f'|sigma_censored - sqrt(SSR/n)|  = {abs(cens.sigma_ - np.sqrt(ols.ssr/n)):.2e}')
print(f'|loglik_censored - loglik_OLS|  = {abs(cens.llf_ - ols.llf):.2e}')

max |beta_censored - beta_OLS|  = 3.00e-05
max |beta_truncated - beta_OLS| = 4.44e-16
|sigma_censored - sqrt(SSR/n)|  = 7.55e-07
|loglik_censored - loglik_OLS|  = 5.90e-07


All discrepancies are at the level of the optimiser tolerance. The estimator reduces to OLS in the no-censoring limit, as it should, so the likelihood and its optimisation are wired up correctly.

## With censoring: agreement with R, disagreement with OLS

The decisive test is a genuinely censored dataset. We compare four numbers per coefficient:

- **true** — the data-generating value;
- **censtrunc** — this package's MLE;
- **R survreg** — R's Gaussian `survreg`, the engine behind `AER::tobit`, computed by an entirely independent codebase;
- **OLS** — naive least squares, which is biased under censoring.

Two correct maximum-likelihood implementations must converge to the *same unique* optimum, so `censtrunc` and R should agree to optimiser tolerance — while both should differ from the biased OLS and sit close to the truth.

In [4]:
import shutil, subprocess, json, tempfile, os
from pathlib import Path

rng2 = np.random.default_rng(20260527)
nc = 4000
Xc = rng2.normal(size=(nc, 2))
y_star_c = 1.0 + 0.7*Xc[:, 0] - 0.4*Xc[:, 1] + rng2.normal(size=nc)
Lc, Rc = 0.0, 2.5
yc = np.clip(y_star_c, Lc, Rc)

cens = CensoredRegression(left=Lc, right=Rc).fit(Xc, yc)
ols_c = np.asarray(sm.OLS(yc, sm.add_constant(Xc)).fit().params)

# Run R's survreg via the bundled reference script, if R is available.
def _find_r_script():
    for c in [Path('tests/reference/fit_tobit.R'),
              Path('../tests/reference/fit_tobit.R')]:
        if c.exists():
            return c
    return None

r_beta = r_sigma = None
rscript, script_path = shutil.which('Rscript'), _find_r_script()
if rscript and script_path:
    fd, csvp = tempfile.mkstemp(suffix='.csv'); os.close(fd)
    np.savetxt(csvp, np.column_stack([yc, Xc]), delimiter=',', header='y,x1,x2', comments='')
    try:
        out = subprocess.run([rscript, str(script_path), csvp, str(Lc), str(Rc)],
                             capture_output=True, text=True, timeout=120, check=True)
        rt = json.loads(out.stdout)['tobit']
        r_beta = np.array([rt['coef']['(Intercept)'], rt['coef']['x1'], rt['coef']['x2']])
        r_sigma = float(rt['scale'])
    except Exception:
        r_beta = None
    finally:
        os.remove(csvp)

In [5]:
if r_beta is not None:
    table = pd.DataFrame({
        'true':         [1.0, 0.7, -0.4],
        'censtrunc':    cens.coef_,
        'R survreg':    r_beta,
        'OLS (biased)': ols_c,
    }, index=['const', 'x1', 'x2'])
    display(table.round(6))
    print(f'max |censtrunc - R|  = {np.max(np.abs(cens.coef_ - r_beta)):.2e}  (independent solvers, same optimum)')
    print(f'|sigma_censtrunc - sigma_R| = {abs(cens.sigma_ - r_sigma):.2e}')
    print(f'max |censtrunc - OLS| = {np.max(np.abs(cens.coef_ - ols_c)):.3f}  (Tobit correction is real)')
else:
    print('R not available at build time.')
    print('The live comparison runs in the test suite: tests/test_r_reference.py')
    print('censtrunc estimates:', cens.coef_.round(4))
    print('OLS (biased):       ', ols_c.round(4))

,true,censtrunc,R survreg,OLS (biased)
const,1.0,1.014284,1.014284,1.090364
x1,0.7,0.701331,0.701331,0.470177
x2,-0.4,-0.410105,-0.410105,-0.277838


max |censtrunc - R|  = 3.05e-08  (independent solvers, same optimum)
|sigma_censtrunc - sigma_R| = 2.41e-08
max |censtrunc - OLS| = 0.231  (Tobit correction is real)


The `censtrunc` and R columns are identical to about six decimals, yet `max |censtrunc - R|` is a tiny *non-zero* number (~1e-8): the two independent optimisers land on the same maximum without being bit-for-bit copies. Both recover the true coefficients, while OLS is visibly biased toward zero — exactly the censoring attenuation predicted by Greene (1981).

## Probit limit: tight thresholds reduce Tobit to probit

Consider the censored regression with $L = R = c$. With no interior region the likelihood reduces to

$$\ell(\beta, \sigma) = \sum_i \mathbf 1\{y_i = c\}\,\log\Phi\!\left(\tfrac{c - X_i'\beta}{\sigma}\right) +\mathbf 1\{y_i > c\}\,\log\!\left[1 - \Phi\!\left(\tfrac{c - X_i'\beta}{\sigma}\right)\right],$$

which is *exactly* the probit log-likelihood for the indicator $D_i = \mathbf 1\{Y^*_i > c\}$, identifying $\beta_{\text{probit}} = \beta_{\text{tobit}}/\sigma_{\text{tobit}}$ (Hansen 2022, §27.4).

We cannot set $L = R$ literally (then $\sigma$ is unidentified), but a *narrow band* $[c - \varepsilon, c + \varepsilon]$ is close enough. A handful of interior observations identifies $\sigma$; the rest of the data act like a binary outcome. So as $\varepsilon\to 0$ we expect $\hat\beta_{\text{tobit}}/\hat\sigma_{\text{tobit}}\to\hat\beta_{\text{probit}}$.

In [6]:
from statsmodels.discrete.discrete_model import Probit

rng3 = np.random.default_rng(20250601)
n3 = 8000
Xp = rng3.normal(size=(n3, 2))
beta_p = np.array([0.4, 1.2, -0.8])
y_star_p = beta_p[0] + Xp @ beta_p[1:] + rng3.normal(size=n3)
y_bin = (y_star_p > 0.0).astype(float)
probit = Probit(y_bin, sm.add_constant(Xp)).fit(disp=False)

def fit_band(eps: float):
    y_obs = np.clip(y_star_p, -eps, eps)
    m = CensoredRegression(left=-eps, right=eps).fit(Xp, y_obs)
    return m.coef_ / m.sigma_   # the ratio that should match probit

table = pd.DataFrame({
    'probit beta':                     np.asarray(probit.params),
    'tobit / sigma  (eps=0.5)':        fit_band(0.50),
    'tobit / sigma  (eps=0.1)':        fit_band(0.10),
    'tobit / sigma  (eps=0.05)':       fit_band(0.05),
}, index=['const', 'x1', 'x2'])
table.round(4)

,probit beta,tobit / sigma (eps=0.5),tobit / sigma (eps=0.1),tobit / sigma (eps=0.05)
const,0.3791,0.3755,0.3728,0.3744
x1,1.2218,1.1923,1.2134,1.2192
x2,-0.7774,-0.7726,-0.7754,-0.7783


As the band shrinks, the tobit-to-probit ratio approaches the probit estimate. The intercept drifts more than the slopes — a finite-band effect that scales like $\varepsilon/\sigma$ — but at $\varepsilon = 0.05$ the slopes agree to within a few percent, which checks the boundary terms of $\ell$ separately from the OLS limit above.